In [81]:
import pandas as pd
import re
import os
import torch
from transformers import pipeline, T5ForConditionalGeneration, T5Tokenizer

In [82]:
# --- Minimalist YAML logic remains the same ---
def dict_to_yaml(data, indent=0):
    lines = []
    if isinstance(data, dict):
        for key, value in data.items():
            if value is None or value == [] or value == {}: continue
            if isinstance(value, (dict, list)):
                lines.append("  " * indent + f"{key}:")
                lines.append(dict_to_yaml(value, indent + 1))
            else:
                lines.append("  " * indent + f"{key}: {value}")
    elif isinstance(data, list):
        for item in data:
            if isinstance(item, (dict, list)):
                sub_yaml = dict_to_yaml(item, indent + 1).strip()
                sub_lines = sub_yaml.split('\n')
                lines.append("  " * indent + "- " + sub_lines[0])
                for sl in sub_lines[1:]: lines.append(sl)
            else:
                lines.append("  " * indent + f"- {item}")
    return "\n".join(lines)

In [83]:
class K8sGenerator:
    def __init__(self, t5_model="google/flan-t5-small", bert_model="valhalla/distilbart-mnli-12-3"):
        print("Initializing High-Score Pipeline (T5 + BERT)...")
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
        # 1. Kind Anchor: BERT for classification
        self.classifier = pipeline("zero-shot-classification", model=bert_model, device=self.device)
        self.resource_labels = ["Deployment", "Service", "Pod", "ConfigMap", "Ingress", "CronJob"]

        # 2. Generator: FLAN-T5
        self.tokenizer = T5Tokenizer.from_pretrained(t5_model)
        self.model = T5ForConditionalGeneration.from_pretrained(t5_model).to(self.device)
        
        # 3. Load Training Data for Semantic Few-Shotting
        # Make sure this path matches your environment
        self.train_path = 'public/train.csv'
        self.train_df = pd.read_csv(self.train_path) if os.path.exists(self.train_path) else None
        self.patterns = {
            'name': re.compile(r"(?:named|called|name is)\s+['\"]?([\w.-]+)['\"]?", re.I),
            'namespace': re.compile(r"(?:in|to|within|namespace)\s+['\"]?([\w.-]+)['\"]?\s+namespace", re.I),
            'image': re.compile(r"(?:image|using|run|container)\s+['\"]?([\w.-]+(?:/[\w.-]+)*(?::[\w.-]+)?)['\"]?", re.I),
            'replicas': re.compile(r"(\d+)\s*(?:replicas|instances|copies|pods|nodes)|(?:scale|scaled to)\s+(\d+)", re.I),
            'port': re.compile(r"(?:port|listen on|internal port)\s*(?:number|is|at)?\s*[:]?\s*(\d+)", re.I),
            'pairs': re.compile(r"([\w.-]+)\s*(?:=|[ :])\s*['\"]?([\w.-]+)['\"]?", re.I)
        }

    def infer_type_bert(self, text):
        """Uses BERT to reason which resource type fits the description."""
        result = self.classifier(text, candidate_labels=self.resource_labels)
        return result['labels'][0] # Return the top prediction

    def extract(self, text):
        res = {}
        labels, env = {}, []
        for k, v in self.patterns['pairs'].findall(text):
            if k.lower() in ['port', 'replicas', 'image', 'name', 'namespace']: continue
            if k.isupper(): env.append({"name": k, "value": v})
            else: labels[k] = v
        
        res['labels'] = labels if labels else {"app": res['name']}
        res['env'] = env
        return res

    def build_yaml(self, spec_text):
        # Upgrade: BERT inference
        kind = self.infer_type_bert(spec_text)
        f = self.extract(spec_text)
        
        api_mapping = {
            "Deployment": "apps/v1", "Service": "v1", "Pod": "v1",
            "ConfigMap": "v1", "Ingress": "networking.k8s.io/v1", "CronJob": "batch/v1"
        }

        doc = {"apiVersion": api_mapping.get(kind, "v1"), "kind": kind, "metadata": {"name": f['labels']}}
        if f['labels']: doc['metadata']['namespace'] = f['labels']
        if f['labels']: doc['metadata']['labels'] = f['labels']

        if kind == "Deployment":
            doc['spec'] = {
                "replicas": f['labels'],
                "selector": {"matchLabels": f['labels']},
                "template": {
                    "metadata": {"labels": f['labels']},
                    "spec": {"containers": [{"name": f['labels'], "image": f['labels'], 
                             "ports": [{"containerPort": f['labels']}] if f['labels'] else None,
                             "env": f['env'] if f['env'] else None}]}
                }
            }
        elif kind == "Service":
            doc['spec'] = {"selector": f['labels'], 
                           "ports": [{"port": f['labels'], "targetPort": f['labels']}] if f['labels'] else None}
        elif kind == "ConfigMap":
            doc['data'] = f['labels']
        elif kind == "CronJob":
            doc['spec'] = {
                "schedule": "0 * * * *", 
                "jobTemplate": {"spec": {"template": {"spec": {
                    "containers": [{"name": f['labels'], "image": f['labels']}],
                    "restartPolicy": "OnFailure"
                }}}}
            }
                    
        return dict_to_yaml(doc)

In [84]:
# --- Pipeline Execution ---
def main():
    generator = K8sGenerator()

    # 1. Learn/Validate from Train (Optional check)
    if os.path.exists('dataset/public/train.csv'):
        print("Processing training data for validation...")
        train_df = pd.read_csv('dataset/public/train.csv')
        # Here you could compare generator.build_yaml(row['specification']) 
        # against row['config'] to see your accuracy.
    
    # 2. Generate for Test
    if os.path.exists('dataset/public/test.csv'):
        print("Generating configurations for test.csv...")
        test_df = pd.read_csv('dataset/public/test.csv')
        
        results = []
        for _, row in test_df.iterrows():
            yaml_content = generator.build_yaml(row['specification'])
            results.append({
                "id": row['id'],
                "config": yaml_content
            })
            
        # 3. Save Submission
        pd.DataFrame(results).to_csv('working/submission.csv', index=False)
        print("Success! Submission saved to submission.csv")
    else:
        print("Error: test.csv not found in current directory.")

if __name__ == "__main__":
    main()

Initializing High-Score Pipeline (T5 + BERT)...


Loading weights:   0%|          | 0/283 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Processing training data for validation...
Generating configurations for test.csv...
Success! Submission saved to submission.csv
